# Jedna partycja per device i domyślny kierunek ASC clustering po event_time
Wszystkie dane urządzenia trafiają do jednej partycji, co sprawia, że odczyt staje się coraz wolniejszy.
Dane posortowane są w kolejności od najstarszych do najmłodszych.
Partition key = device_id
Clustering key = event_time ASC

In [1]:
import uuid, time
from cassandra.cluster import Cluster

In [2]:
# Otwieranie połączenia
cluster = Cluster(['cassandra_nosql_lab'], port=9042)
session = cluster.connect()
print("Połączono z Cassandra")

Połączono z Cassandra


In [3]:
# Tworzenie Keyspace
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS lab_keyspace
    WITH replication = {'class': 'SimpleStrategy', 'replication_factor': '1'}
""")
print("Utworzono keyspace")

Utworzono keyspace


In [4]:
# Użycie Keyspace
session.set_keyspace('lab_keyspace')

# Tworzenie tabeli
session.execute("""
    CREATE TABLE IF NOT EXISTS events_by_device_2 (
        device_id TEXT,
        day TEXT,
        event_time TIMESTAMP,
        temperature FLOAT,
        humidity FLOAT,
        PRIMARY KEY (device_id, event_time)
    )
""")
print("Utworzono tabelę 'users'")

Utworzono tabelę 'users'


## Generate test data

In [5]:
!python ../_data_generator/main.py \
  --loader cassandra \
  --table events_by_device_2 \
  --records 100000 \
  --batch-size 100

Done. Loader=cassandra, Records=100000


In [6]:
start = time.perf_counter()
rows = session.execute("""
SELECT *
FROM
    events_by_device_2
WHERE
    device_id='device_1'
LIMIT 10
""")
end = time.perf_counter()
print(f"Zapytanie wykonane w {end - start:.4f} sekund")
print("Znalezione rekordy:")
for row in rows:
    print(f"- {row.device_id} {row.day} ({row.temperature}°C, {row.humidity}%) - {row.event_time}")

Zapytanie wykonane w 0.0111 sekund
Znalezione rekordy:
- device_1 2026-03-28 (29.049999237060547°C, 36.25%) - 2026-03-27 16:03:50.286000
- device_1 2026-03-28 (20.399999618530273°C, 61.9900016784668%) - 2026-03-27 16:04:17.502000
- device_1 2026-03-28 (28.170000076293945°C, 69.86000061035156%) - 2026-03-27 16:04:35.141000
- device_1 2026-03-28 (34.439998626708984°C, 64.02999877929688%) - 2026-03-27 16:05:29.907000
- device_1 2026-03-28 (28.31999969482422°C, 39.68000030517578%) - 2026-03-27 16:05:35.123000
- device_1 2026-03-28 (28.100000381469727°C, 30.579999923706055%) - 2026-03-27 16:06:17.468000
- device_1 2026-03-28 (21.969999313354492°C, 65.0999984741211%) - 2026-03-27 16:06:35.123000
- device_1 2026-03-28 (25.549999237060547°C, 59.849998474121094%) - 2026-03-27 16:07:08.763000
- device_1 2026-03-28 (33.59000015258789°C, 42.939998626708984%) - 2026-03-27 16:07:10.743000
- device_1 2026-03-28 (32.0°C, 46.77000045776367%) - 2026-03-27 16:07:10.745000


In [7]:
# Zamknięcie połączenia
cluster.shutdown()